# 04 — 訓練 MNIST（Pipeline Editor 第 1 步）

用 **PyTorch CNN** 訓練手寫數字分類，匯出 **ONNX** 到固定路徑。

| 用途 | 說明 |
|------|------|
| **單元 B-Vis** | Pipeline Editor **第一個節點**；後方接 **`05-validate-mnist-onnx.ipynb`** |
| Workbench 手動 | 也可在此 Notebook 逐格 Run（較慢，且寫入 Workbench PVC） |

**Data Volume（Elyra 常忽略 Sub path）**：把整個 `model-storage` 掛到：
```
/opt/app-root/src/model-storage
```

**產出路徑**（相對 PVC，給 Dashboard Deploy）：
```
mnist-onnx/1/model.onnx
```
完整容器路徑：
```
/opt/app-root/src/model-storage/mnist-onnx/1/model.onnx
```

> **OVMS 必要**：需要版本目錄 **`1/`**。  
> **Deploy Model path**：填 **`mnist-onnx/`**（**不要**填 `/`，Dashboard 會無法 Next；PVC 根請改填 `.`）。

**映像**：請用 **Jupyter | Data Science | CPU | Python 3.12**（含 Elyra）。

教學預設為 **CPU、小資料子集**（本包學員主線 B-Vis）。


## 安裝依賴

Data Science 映像預設可能沒有 PyTorch；Pipeline Pod 每次 Run 也會執行此格。


In [ ]:
# CPU 版 PyTorch（課堂用；氣隙環境請改用講師預先準備的套件來源）
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install -q onnx onnxscript


In [ ]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# 整個 model-storage PVC 掛在此（Sub path 留空；Elyra 常忽略 subPath）
MOUNT = Path("/opt/app-root/src/model-storage")
# Dashboard Deploy Model path = mnist-onnx/  →  PVC 內這層
MODEL_DIR = MOUNT / "mnist-onnx"
# OVMS：<model_dir>/<version>/model.onnx
VERSION_DIR = MODEL_DIR / "1"
VERSION_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = VERSION_DIR / "model.onnx"
PTH_PATH = MODEL_DIR / "mnist_model.pth"

# 課堂加速：預設 1 epoch、最多 8000 筆訓練樣本（可用環境變數覆寫）
EPOCHS = int(os.environ.get("EPOCHS", "1"))
MAX_SAMPLES = int(os.environ.get("MAX_SAMPLES", "8000"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "64"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, EPOCHS={EPOCHS}, MAX_SAMPLES={MAX_SAMPLES}")
print(f"ONNX_PATH={ONNX_PATH}")


In [ ]:
class Net(nn.Module):
    """與 training/pytorch-mnist/scripts/train.py 相同結構（單機版）。"""

    def __init__(self) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST("/tmp/data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST("/tmp/data", train=False, download=True, transform=transform)

if MAX_SAMPLES and MAX_SAMPLES < len(train_ds):
    train_ds = Subset(train_ds, list(range(MAX_SAMPLES)))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

model = Net().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(f"train samples={len(train_ds)}, test samples={len(test_ds)}")


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Batch {batch_idx}, Loss: {loss.item():.4f}")
    print(f"Epoch {epoch+1} avg loss: {total_loss / max(len(train_loader), 1):.4f}")

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)
acc = correct / total
print(f"Test accuracy: {acc:.4f}")


In [ ]:
torch.save(model.state_dict(), PTH_PATH)
print(f"PyTorch weights saved: {PTH_PATH}")

export_model = model.cpu().eval()
dummy = torch.randn(1, 1, 28, 28)
torch.onnx.export(
    export_model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=18,
)
print(f"ONNX model saved: {ONNX_PATH} ({ONNX_PATH.stat().st_size} bytes)")
assert ONNX_PATH.exists(), "ONNX 未成功寫出"
print("Training completed successfully.")


## 下一步：Pipeline 第二步 + 部署

1. 在 Pipeline Editor 再拖入 **`05-validate-mnist-onnx.ipynb`**，從本節點拉線連過去（或開 `mnist-train-pipeline.pipeline`）
2. 確認 **04、05 都 Succeeded**
3. Dashboard → **Deploy model**

| 欄位 | 值 |
|------|-----|
| Name | `mnist-classifier-elyra` |
| Framework | **ONNX** |
| Cluster storage | `model-storage` |
| Model path | **`mnist-onnx/`**（勿填 `/`） |

見入門教學單元 B。
